In [ ]:
# %pip install -r requirements.txt

In [28]:
import numpy as np
import librosa
import soundfile as sf

In [29]:
audio_path = 'audio/'

In [30]:
A_audio, sr = librosa.load(audio_path + 'Synth.wav', sr=None)
B_audio, _ = librosa.load(audio_path + 'Piano.wav', sr=None)
C_audio, _ = librosa.load(audio_path + 'BlindingLights.wav', sr=None)


In [31]:
n_fft = 1024
hop_length = n_fft // 4

In [32]:
MA_complex = librosa.stft(A_audio, n_fft=n_fft, hop_length=hop_length)
MB_complex = librosa.stft(B_audio, n_fft=n_fft, hop_length=hop_length)
MC_complex = librosa.stft(C_audio, n_fft=n_fft, hop_length=hop_length)


In [33]:
MA = np.abs(MA_complex)
MB = np.abs(MB_complex)
MC = np.abs(MC_complex)

In [34]:
min_frames = min(MA.shape[1], MB.shape[1])

MA = MA[:, :min_frames]
MB = MB[:, :min_frames]

In [35]:
MA_pinv = np.linalg.pinv(MA)

In [18]:
T = np.dot(MB, MA_pinv)

In [20]:
fro_norm_squared = np.linalg.norm(np.dot(T, MA) - MB, 'fro') ** 2
print('Squared Frobenius norm:', fro_norm_squared)

Squared Frobenius norm: 49536.22


In [36]:
np.savetxt('problem3t.csv', T, delimiter=',')

In [37]:
MD = np.dot(T, MC)

In [38]:
np.savetxt('problem3md.csv', MD, delimiter=',')

In [39]:
MC_phase = np.angle(MC_complex)
MD_complex = MD * np.exp(1j * MC_phase)

In [40]:
D_audio = librosa.istft(MD_complex, hop_length=hop_length)
sf.write('problem3.wav', D_audio, sr)